In [1]:
import xarray as xr
import rioxarray as rio
import glob
from pathlib import Path
import natsort
import subprocess
import numpy as np


In [ ]:
# get all the speedup 
files = natsort.natsorted(glob.glob('windmapper_config/*.tif'))

# the shortest name is the the base DEM. Remove this basename from the variable name used in the dataset
basename = Path(min(files, key=len)).stem

variables = []
dem = None
for tif_file in files:
    # Use the file name (without extension) as the variable name
    variable_name = Path(tif_file).stem

    if variable_name == basename:

        # The WM area is smaller than the input DEM due to the strict alignment requirements
        # cut down the input DEM to match the output WM extent
        variable_name = variable_name.replace(basename,'DEM')
        shp = Path(tif_file).parent / 'shp' / 'user_bbox.shp'
        new_dem = 'DEM-WM-clip.tif'

        command = [
            "gdalwarp",  # Replace with actual path to gdalwarp
            "-cutline", shp,
            "-crop_to_cutline",
            "-dstalpha",
            "-overwrite",
            'FABDEM-clip.tif',
            new_dem
        ]
        result = subprocess.run(command, capture_output=True, text=True)
        dem = xr.open_mfdataset(new_dem).isel(band=0).drop_vars('band')
        dem = dem.rename({'band_data': 'DEM','x':'lon','y':'lat'})
        continue
        
    else:
        variable_name = variable_name.replace(basename,'dir')
    
    ds = xr.open_mfdataset(tif_file).isel(band=0).drop_vars('band')
    ds = ds.rename({'band_data': variable_name,'x':'lon','y':'lat'})
    variables.append(ds)

# Combine all variables into a single xarray.Dataset
# Enforce them to all /exactly/ match the same x,y and compensate for any tiny floating point differences in the lat lon coords 
ds = xr.merge( xr.align(*variables, join="override"), compat='no_conflicts', join='exact')
ds


In [3]:
# Create a new dataset with the same coordinates as the existing dem
wind = xr.Dataset(coords=dem.coords)

# Add a new variable to the new dataset (optional)
wind["speed"] = (("lon", "lat"), np.zeros((557,406))+10) # a constant 10m/s wind
wind["dir"] = (("lon", "lat"), np.zeros((557,406)) + 45) # a 45 degree wind

# where the output will go
wind["u"] = (("lon", "lat"), np.zeros((557,406))) 
wind["v"] = (("lon", "lat"), np.zeros((557,406))) 

In [ ]:
# example usage. The tight loop + lookup is very very slow

# this would be run for each timestep of the using model
N_windfield = len(ds.data_vars) // 3
delta_angle = 360. / N_windfield

# Iterate over intput wind field
for i in range(wind.sizes['lon']):  
    print(i, wind.sizes['lon'])
    lon = wind['lon'][i]
    for j in range(wind.sizes['lat']):
        
        lat = wind['lat'][j]

        # find the nearest point in the windmapper output because the using DEM and WM output don't have to be 1:1 exactly
        nearest_point = ds.sel(lat=lat, 
                            lon=lon, 
                            method="nearest")
        nearest_lat = nearest_point["lat"].values
        nearest_lon = nearest_point["lon"].values

        lon_i, lat_j = np.where((ds.lat == nearest_point.lat) & (ds.lon == nearest_point.lon))

        # get the input wind dir
        theta = wind.dir[i,j].values.flatten()[0]

        # compute the two two nearest wind maps to this input direction
        d1 = int(theta / delta_angle)
        theta1 = d1 * delta_angle 
        if d1 == 0:
            d1 = N_windfield

        d2 = int((theta  + delta_angle) / delta_angle)
        theta2 = d2 * delta_angle 
        if d2 == 0:
            d2 = N_windfield

        d1 = int(d1 * delta_angle)
        d2 = int(d2 * delta_angle)

        if d1 == 360:
            d1 = 0
        if d2 == 360:
            d2 = 0

        U_d1 = ds[f'dir_{d1}_U'][lon_i,lat_j].values.flatten()[0]
        U_d2 = ds[f'dir_{d2}_U'][lon_i,lat_j].values.flatten()[0]

        V_d1 = ds[f'dir_{d1}_V'][lon_i,lat_j].values.flatten()[0]
        V_d2 = ds[f'dir_{d2}_V'][lon_i,lat_j].values.flatten()[0]

        W_d1 = ds[f'dir_{d1}_spd_up_1000'][lon_i,lat_j].values.flatten()[0]
        W_d2 = ds[f'dir_{d2}_spd_up_1000'][lon_i,lat_j].values.flatten()[0]

        # interpolate between them
        U = U_d1*(theta2-theta)/(theta2-theta1)+U_d2*(theta-theta1)/(theta2-theta1)
        V = V_d1*(theta2-theta)/(theta2-theta1)+V_d2*(theta-theta1)/(theta2-theta1)
        W_transf = W_d1*(theta2-theta)/(theta2-theta1)+W_d2*(theta-theta1)/(theta2-theta1)

        # update the direction
        wind.u[i,j] = U
        wind.v[i,j] = V

        # update the speed
        wind.speed[i,j] = wind.speed[i,j] * W_transf

        # TODO: fix degree / radian 
        # sign convention 
        # http://mst.nerc.ac.uk/wind_vect_convs.html
        # new_dir = np.arctan2(-U, -V)

        # if new_dir < 0:
        #     new_dir = new_dir + 360.

        # wind.dir[i,j] = new_dir


In [ ]:
# same as previous cell but a bit more pre computed & vectorized
# still comically slow

# precompute static fields
N_windfield = len(ds.data_vars) // 3
delta_angle = 360. / N_windfield

lon_grid, lat_grid = np.meshgrid(wind['lon'], wind['lat'], indexing='ij')
nearest_points = ds.sel(lat=xr.DataArray(lat_grid, dims=["lon", "lat"]),
                        lon=xr.DataArray(lon_grid, dims=["lon", "lat"]),
                        method="nearest")

theta_grid = wind['dir']
# Compute angle bins (d1, d2) and their corresponding angles (theta1, theta2)
d1 = np.floor(theta_grid / delta_angle).astype(int)
d2 = (d1 + 1) % N_windfield
theta1_grid = d1 * delta_angle
theta2_grid = d2 * delta_angle

# Handle edge cases where d1 or d2 wrap around to 0 or 360 degrees
d1 = xr.where(d1 == 0, N_windfield, d1)
d2 = xr.where(d2 == 0, N_windfield, d2)

# Convert angle bins to dataset variable names
d1_angles = (d1 * delta_angle).astype(int)
d2_angles = (d2 * delta_angle).astype(int)

d1_angles = xr.where(d1_angles == 360, 0, d1_angles)
d2_angles = xr.where(d2_angles == 0, 0, d2_angles)


for i in range(wind.sizes['lon']):  # Iterate over the first dimension (x)
    print(i, wind.sizes['lon'])
    for j in range(wind.sizes['lat']):
        
        U_d1 = nearest_points[f'dir_{d1_angles[i,j].values.flatten()[0]}_U'][i,j].values.flatten()[0]
        U_d2 = nearest_points[f'dir_{d2_angles[i,j].values.flatten()[0]}_U'][i,j].values.flatten()[0]

        V_d1 = nearest_points[f'dir_{d1_angles[i,j].values.flatten()[0]}_V'][i,j].values.flatten()[0]
        V_d2 = nearest_points[f'dir_{d2_angles[i,j].values.flatten()[0]}_V'][i,j].values.flatten()[0]

        W_d1 = nearest_points[f'dir_{d1_angles[i,j].values.flatten()[0]}_spd_up_1000'][i,j].values.flatten()[0]
        W_d2 = nearest_points[f'dir_{d2_angles[i,j].values.flatten()[0]}_spd_up_1000'][i,j].values.flatten()[0]

        theta = theta_grid[i,j].values.flatten()[0]
        theta1 = theta1_grid[i,j].values.flatten()[0]
        theta2 = theta2_grid[i,j].values.flatten()[0]

        U = U_d1*(theta2-theta)/(theta2-theta1)+U_d2*(theta-theta1)/(theta2-theta1)
        V = V_d1*(theta2-theta)/(theta2-theta1)+V_d2*(theta-theta1)/(theta2-theta1)
        W_transf = W_d1*(theta2-theta)/(theta2-theta1)+W_d2*(theta-theta1)/(theta2-theta1)

        wind.u[i,j] = U
        wind.v[i,j] = V

        wind.speed[i,j] = wind.speed[i,j] * W_transf


In [ ]:
# WIP vectorized approach, ignore
# Precompute constants
N_windfield = len(ds.data_vars) // 3
delta_angle = 360.0 / N_windfield

# Precompute nearest neighbors for all lat/lon combinations
lon_grid, lat_grid = np.meshgrid(wind['lon'], wind['lat'], indexing='xy')
nearest_points = ds.sel(lat=xr.DataArray(lat_grid, dims=["lon", "lat"]),
                        lon=xr.DataArray(lon_grid, dims=["lon", "lat"]),
                        method="nearest")

nearest_lats = nearest_points["lat"].values
nearest_lons = nearest_points["lon"].values

# Precompute indices for nearest neighbors
lon_indices = np.searchsorted(ds['lon'], nearest_lons)
lat_indices = np.searchsorted(ds['lat'], nearest_lats)

# Vectorized computation over the entire grid
theta = wind['dir'].values  # Shape (lon, lat)



# Extract U, V, and W components for d1 and d2 using precomputed indices
U_d1 = ds[f'dir_{d1_angles}_U'].values[lon_indices, lat_indices]
U_d2 = ds[f'dir_{d2_angles}_U'].values[lon_indices, lat_indices]

V_d1 = ds[f'dir_{d1_angles}_V'].values[lon_indices, lat_indices]
V_d2 = ds[f'dir_{d2_angles}_V'].values[lon_indices, lat_indices]

W_d1 = ds[f'dir_{d1_angles}_spd_up_1000'].values[lon_indices, lat_indices]
W_d2 = ds[f'dir_{d2_angles}_spd_up_1000'].values[lon_indices, lat_indices]

# Interpolate U, V, and W components based on theta
U = U_d1 * (theta2 - theta) / (theta2 - theta1) + U_d2 * (theta - theta1) / (theta2 - theta1)
V = V_d1 * (theta2 - theta) / (theta2 - theta1) + V_d2 * (theta - theta1) / (theta2 - theta1)
W_transf = W_d1 * (theta2 - theta) / (theta2 - theta1) + W_d2 * (theta - theta1) / (theta2 - theta1)

# Update wind dataset with computed values
wind['u'].values[:, :] = U
wind['v'].values[:, :] = V
wind['speed'].values[:, :] *= W_transf